## M0 Setup

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'config.yaml').exists() and (ROOT.parent / 'config.yaml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from m0_setup.config import load_config

print('environment ready')

## Parameters edit here, then Run All

These override `config.yaml` for this notebook run. Any key you leave out keeps
the value from `config.yaml`. Changing the dates or the stock list refreshes the
raw snapshot automatically (the cache notices the change). Anything you change
flows through every stage below.

Notes:
- `n_folds × test_size` trading days are held out, oldest first.
- `explain_sample_size` caps how many test rows SHAP explains per fold.
- A full run is a few minutes; the LSTM is not part of the MVP.

In [ ]:
# ---- tunable parameters: edit here, then Run All ----
# Any key left out keeps its value from config.yaml.
PARAMS = {
    'seed': 42,
    'data': {
        'start_date': '2015-01-01',
        'end_date': '2026-09-19',
        'stocks': ['BHP', 'CBA', 'CSL', 'NAB', 'WBC', 'ANZ', 'WES', 'RIO', 'MQG', 'TLS'],
        'force_download': False,
    },
    'features': {
        'rsi_window': 14,
        'sma_windows': [10, 20, 50],
        'ema_windows': [12, 26],
        'return_lags': [1, 2, 3, 5],
    },
    'walk_forward': {'n_folds': 8, 'val_size': 60, 'test_size': 120, 'min_train_size': 500},
    'models': {
        'logistic': {'C': 1.0, 'max_iter': 1000, 'class_weight': 'balanced'},
        'xgboost': {
            'n_estimators': 300,
            'max_depth': 4,
            'learning_rate': 0.05,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'reg_lambda': 1.0,
        },
    },
    'shap': {'explain_sample_size': 2000},
    'consistency': {'rolling_window': 120, 'diagnostic_window': 60, 'top_k': 10},
    'faithfulness': {'top_k': 5},
}

cfg = load_config(ROOT / 'config.yaml', overrides=PARAMS)
print('effective parameters')
print('  seed:', cfg['seed'], '| stocks:', len(cfg['data']['stocks']), '| folds:', cfg['walk_forward']['n_folds'])
print('  dates:', cfg['data']['start_date'], '->', cfg['data']['end_date'])
print('  xgboost:', cfg['models']['xgboost'])
print('  results dir:', cfg['paths']['results'])

## M1  Data acquisition (SP1)

In [ ]:
from m1_data.data_collection import run as run_data

coverage = run_data(cfg)
coverage

## M2  Preprocessing, features and target (SP2)

In [ ]:
from m2_features.features import run as run_features, feature_columns

table = run_features(cfg)
print('rows:', len(table), '| features:', len(feature_columns(cfg)))
table.head(3)

## M3  Chronological walk-forward folds (SP3a)

In [ ]:
from m3_walkforward.validation import walk_forward_folds

folds = walk_forward_folds(table['date'], cfg)
pd.DataFrame([
    {
        'fold': fold.index,
        'train_days': len(fold.train),
        'val_days': len(fold.val),
        'test_start': pd.Timestamp(fold.test.min()).date(),
        'test_end': pd.Timestamp(fold.test.max()).date(),
    }
    for fold in folds
])

## M4  LogReg + XGBoost and walk-forward metrics (SP3b)

In [ ]:
from m4_models.models import run as run_models

predictions = run_models(cfg)
print('\nPooled metrics')
display(pd.read_csv(Path(cfg['paths']['tables']) / 'model_metrics_pooled.csv').round(4))
print('Naive baselines')
display(pd.read_csv(Path(cfg['paths']['tables']) / 'baseline_metrics.csv').round(4))

## M6  SHAP attributions (SP4)

In [ ]:
from m6_shap.explainability import run as run_shap

shap_frames = run_shap(cfg)
importance = pd.read_csv(Path(cfg['paths']['tables']) / 'shap_global_importance.csv')
print('\nTop 10 by mean |SHAP| — XGBoost')
display(importance[importance['model'] == 'xgboost'].head(10)[['rank', 'feature', 'mean_abs_shap']].round(4))

## M7  Financial groups and direction of effect (SP5)

In [ ]:
from m7_interpretation.interpretation import run as run_interpretation

interpretation = run_interpretation(cfg)
interpretation['group_importance'][['model', 'group', 'share_pct', 'rank']].round(2)

## M8  Two-level consistency, diagnostic and abstention (SP6a/SP6b)

In [ ]:
from m8_consistency.consistency import run as run_consistency

consistency = run_consistency(cfg)
consistency['summary'].round(4)

In [ ]:
from m8_consistency.diagnostic import run as run_diagnostic

diagnostic = run_diagnostic(cfg)
diagnostic['summary'].round(4)

## M9  Faithfulness: permutation and placebo (SP6)

In [ ]:
from m9_faithfulness.faithfulness import run as run_faithfulness

faithfulness = run_faithfulness(cfg)
print('\nPlacebo rank among all features (lower is more faithful)')
display(faithfulness['placebo'].round(4))

## M10  Decision-support prototype (SP7)

In [ ]:
from m10_prototype.prototype import explain_stock, interpretation_text

prediction = explain_stock('BHP', cfg)
print(interpretation_text(prediction))
prediction.group_shares.round(1).to_frame('group_share_pct')